# TSP_DRL_PtrNet — Independent training-seed extension (v1.2)

This notebook trains **two new independent models** using the same pinned source revision and configuration as `clean_restart_seed20261013_v1`.

- Additional training seeds: `20261014`, `20261015`.
- Same architecture, batch size, city count, optimizer, learning rate, and 13,000-update budget as the principal run.
- Each run gets a distinct immutable folder and a complete training manifest.
- The existing `clean_restart_seed20261013_v1` run is an input reference only and is never modified.
- This notebook does not run the benchmark. Use the matching v1.2 evaluation notebook after a run has completed.

**Resource note:** this adds two complete 13,000-update GPU runs, in sequence, plus a small smoke test. Do not interpret the prepared notebook as evidence that the runs have already been executed.


## Step 0 — Configure Colab and lock the run identities

Select a GPU runtime. Run cells from top to bottom. The setup refuses to reuse existing output folders, so an interrupted or completed run cannot be silently overwritten. The new runs are not resumable from actor-only snapshots.


In [ ]:
from pathlib import Path
from datetime import datetime
import os, sys, json, shutil, subprocess, platform, hashlib, re, difflib, time
import numpy as np

try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("Open this notebook in Google Colab, then select a GPU runtime.") from exc

drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/PFE_TSP_DRL')
RUNS_ROOT = PROJECT_ROOT / 'runs'
MAIN_RUN_ID = 'clean_restart_seed20261013_v1'
TRAIN_RUNS = [
    {'run_id': 'clean_restart_seed20261014_v1', 'seed': 20261014},
    {'run_id': 'clean_restart_seed20261015_v1', 'seed': 20261015},
]
RUN_ROOTS = {r['run_id']: RUNS_ROOT / r['run_id'] for r in TRAIN_RUNS}
SHARED_ROOT = RUNS_ROOT / 'multiseed_extension_20261025_v1'

TRAIN_SEEDS = [r['seed'] for r in TRAIN_RUNS]
SMOKE_SEED = 424242
N_TEST_INSTANCES = 100
TRAIN_CITY_COUNT = 20
TRAIN_BATCH_SIZE = 512
TRAIN_STEPS = 13000
CHECKPOINT_EVERY = 1000
VALIDATION_SEED = 7302026
FINAL_TEST_SEED = 8302026

MAIN_RUN_ROOT = PROJECT_ROOT / 'runs' / MAIN_RUN_ID
required_main_artifacts = [
    MAIN_RUN_ROOT / 'manifest_pre.json',
    MAIN_RUN_ROOT / 'manifest_training.json',
    MAIN_RUN_ROOT / 'pip-freeze.txt',
    MAIN_RUN_ROOT / 'full_train_n20' / 'Pt' / 'train20_1013_17_14_step12999_act.pt',
]
missing_main = [str(path) for path in required_main_artifacts if not path.is_file()]
if missing_main:
    raise FileNotFoundError('Original clean-restart artifacts are incomplete; no new folders were created. Missing: ' + ', '.join(missing_main))
if SHARED_ROOT.exists():
    raise RuntimeError(f'Shared output folder already exists: {SHARED_ROOT}. Inspect it and use a fresh extension ID rather than overwriting it.')
for run_id, root in RUN_ROOTS.items():
    if root.exists():
        raise RuntimeError(f'Run folder already exists: {root}. Do not overwrite or merge runs. Change the run ID explicitly if this is a new attempt.')

SHARED_ROOT.mkdir(parents=True, exist_ok=False)
print('Project root:', PROJECT_ROOT)
print('Main run (read-only reference):', PROJECT_ROOT / 'runs' / MAIN_RUN_ID)
print('New runs:')
for r in TRAIN_RUNS:
    print(' ', r['run_id'], 'seed=', r['seed'], 'path=', RUN_ROOTS[r['run_id']])
print('Shared patch/environment record:', SHARED_ROOT)
print('Configured updates per new model:', TRAIN_STEPS)


In [ ]:
import torch
print('Python:', sys.version.replace('\n', ' '))
print('Platform:', platform.platform())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA runtime reported by PyTorch:', torch.version.cuda)
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU memory (GiB):', round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))
    subprocess.run(['nvidia-smi'], check=False)
else:
    raise RuntimeError('GPU is not available. In Colab, select Runtime → Change runtime type → GPU, then rerun this cell.')

## Step 1 — Check out the exact source revision used by the main run

The commit is read from the saved main-run manifest; it is not re-resolved to the repository's moving default branch. The new runs must use the same source revision for a meaningful training-seed comparison.


In [ ]:
REPO_URL = 'https://github.com/Rintarooo/TSP_DRL_PtrNet.git'
main_manifest = json.loads((PROJECT_ROOT / 'runs' / MAIN_RUN_ID / 'manifest_pre.json').read_text(encoding='utf-8'))
COMMIT_SHA = main_manifest.get('commit_sha')
if not COMMIT_SHA:
    raise ValueError('The main-run manifest does not contain commit_sha.')

REPO_DIR = Path('/content/TSP_DRL_PtrNet_multiseed')
if not (REPO_DIR / '.git').exists():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--all', '--tags'], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', COMMIT_SHA], check=True)
actual_commit = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
if actual_commit != COMMIT_SHA:
    raise RuntimeError(f'Source commit mismatch: expected {COMMIT_SHA}, got {actual_commit}')
for required in ['actor.py', 'critic.py', 'env.py', 'data.py', 'config.py', 'train.py']:
    if not (REPO_DIR / required).is_file():
        raise FileNotFoundError(f'Expected upstream file is missing: {required}')

manifest_by_run_id = {}
for run in TRAIN_RUNS:
    run_id, seed = run['run_id'], int(run['seed'])
    run_root = RUN_ROOTS[run_id]
    run_root.mkdir(parents=True, exist_ok=False)
    manifest = {
        'project': 'TSP_DRL_PtrNet independent training-seed extension',
        'repository_url': REPO_URL,
        'commit_sha': COMMIT_SHA,
        'run_id': run_id,
        'train_seed': seed,
        'validation_seed': VALIDATION_SEED,
        'final_test_seed': FINAL_TEST_SEED,
        'n_final_test_instances_per_size': N_TEST_INSTANCES,
        'city_sizes': [20, 50, 100],
        'train_city_count': TRAIN_CITY_COUNT,
        'train_batch_size': TRAIN_BATCH_SIZE,
        'train_steps_target': TRAIN_STEPS,
        'checkpoint_every': CHECKPOINT_EVERY,
        'created_at_local': datetime.now().isoformat(timespec='seconds'),
        'extension_of': MAIN_RUN_ID,
        'training_protocol_note': 'Fresh random initialization; no actor checkpoint loaded from any other run.'
    }
    manifest_by_run_id[run_id] = manifest
    (run_root / 'manifest_pre.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')

(SHARED_ROOT / 'git-commit.txt').write_text(COMMIT_SHA + '\n', encoding='utf-8')
print('Pinned source repository:', REPO_URL)
print('Exact commit:', COMMIT_SHA)
print('Prepared independent manifests:', list(manifest_by_run_id))


## Step 2 — Record the runtime environment

The actual Python, PyTorch, CUDA, GPU and dependency versions are recorded for both runs. The new runs are intended to differ by training seed, not by source code or hyperparameters.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tqdm', 'matplotlib', 'numpy', 'pandas'], check=True)
print('Python:', sys.version.replace('\n', ' '))
print('Platform:', platform.platform())
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU is not available. Select Runtime → Change runtime type → GPU, then rerun this cell.')
print('CUDA runtime:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory (GiB):', round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))
freeze = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
main_freeze_path = PROJECT_ROOT / 'runs' / MAIN_RUN_ID / 'pip-freeze.txt'
if not main_freeze_path.is_file():
    raise FileNotFoundError(f'Main-run environment lock is missing: {main_freeze_path}')
main_freeze = main_freeze_path.read_text(encoding='utf-8')
def pinned_package_spec(freeze_text, package_name):
    pattern = re.compile(r'^' + re.escape(package_name) + r'==(.+)$', re.IGNORECASE)
    for line in freeze_text.splitlines():
        match = pattern.match(line.strip())
        if match:
            return match.group(1)
    return None
main_torch_spec = pinned_package_spec(main_freeze, 'torch')
current_torch_spec = pinned_package_spec(freeze, 'torch')
if main_torch_spec and current_torch_spec and main_torch_spec != current_torch_spec:
    raise RuntimeError(
        f'Torch version differs from the main run: main={main_torch_spec}, current={current_torch_spec}. '
        'Do not train the additional seeds in a different PyTorch build; select a matching runtime/environment first.'
    )
print('Main-run torch package:', main_torch_spec, '| current torch package:', current_torch_spec)
(SHARED_ROOT / 'pip-freeze.txt').write_text(freeze, encoding='utf-8')
for run in TRAIN_RUNS:
    run_root = RUN_ROOTS[run['run_id']]
    (run_root / 'pip-freeze.txt').write_text(freeze, encoding='utf-8')
    (run_root / 'git-commit.txt').write_text(COMMIT_SHA + '\n', encoding='utf-8')
    env_record = {
        'python': sys.version,
        'platform': platform.platform(),
        'pytorch': torch.__version__,
        'cuda_runtime': torch.version.cuda,
        'gpu': torch.cuda.get_device_name(0),
        'gpu_memory_gib': round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 3),
        'source_commit': COMMIT_SHA,
    }
    (run_root / 'environment.json').write_text(json.dumps(env_record, indent=2), encoding='utf-8')
print('Environment records saved for both runs.')


## Step 3 — Apply and archive the minimal fixed-budget training patch

The patch starts from the pristine pinned `train.py`; it seeds the random generators, disables the original cumulative-training-cost early stop, and saves periodic actor-only snapshots. It is compiled before any training.


In [ ]:
# Always patch the pristine file from the checked-out Git commit, not a possibly
# partially edited train.py. This also makes rerunning this cell safe.
train_path = REPO_DIR / 'train.py'
upstream_train = subprocess.check_output(
    ['git', '-C', str(REPO_DIR), 'show', 'HEAD:train.py'], text=True
)
(SHARED_ROOT / 'train.py.upstream').write_text(upstream_train, encoding='utf-8')
patched_train = upstream_train

# Add os import if absent. Work line-by-line rather than matching one exact block.
lines = patched_train.splitlines(keepends=True)
if not any(line.strip() == 'import os' for line in lines):
    import_torch_idx = next((i for i, line in enumerate(lines) if line.strip() == 'import torch'), None)
    if import_torch_idx is None:
        raise RuntimeError('Could not find "import torch" in upstream train.py.')
    lines.insert(import_torch_idx, 'import os\n')
patched_train = ''.join(lines)

# Disable only the upstream cumulative-training-cost early-stop block. The repository
# does not use a held-out validation set for this condition, so our planned run uses
# the configured fixed number of steps. Locate the block by its code, not indentation.
lines = patched_train.splitlines(keepends=True)
stop_matches = [i for i, line in enumerate(lines)
                if re.search(r'^\s*if\s*\(\s*ave_L\s*/\s*\(\s*i\s*\+\s*1\s*\)\s*<\s*min_L\s*\)\s*:', line)]
if len(stop_matches) != 1:
    preview = '\n'.join(f'{i+1}: {line.rstrip()}' for i, line in enumerate(lines)
                        if 'ave_L' in line or 'early stop' in line or 't1 = time()' in line)
    raise RuntimeError(f'Expected one cumulative-mean early-stop block; found {len(stop_matches)}. Relevant lines:\n{preview}')
stop_start = stop_matches[0]
indent = lines[stop_start][:len(lines[stop_start]) - len(lines[stop_start].lstrip(' \t'))]
stop_end = None
for j in range(stop_start + 1, len(lines)):
    candidate = lines[j]
    candidate_indent = candidate[:len(candidate) - len(candidate.lstrip(' \t'))]
    if candidate.strip() == 't1 = time()' and candidate_indent == indent:
        stop_end = j
        break
if stop_end is None:
    raise RuntimeError('Found the early-stop condition but could not locate its matching t1 = time() boundary.')
newline = '\r\n' if (lines[stop_start].endswith('\r\n')) else '\n'
replacement = [
    indent + '# Fixed-step experiment patch: disable cumulative-mean early stopping.\n',
    indent + 'current_mean_cost = ave_L / (i + 1)\n',
    indent + 'if current_mean_cost < min_L:\n',
    indent + '\tmin_L = current_mean_cost\n',
]
lines = lines[:stop_start] + replacement + lines[stop_end:]
patched_train = ''.join(lines)

# Set Python, NumPy and PyTorch RNG seeds before constructing the training environment.
lines = patched_train.splitlines(keepends=True)
main_matches = [i for i, line in enumerate(lines)
                if line.strip() == 'cfg = load_pkl(pkl_parser().path)']
if len(main_matches) != 1:
    raise RuntimeError(f'Expected one train.py main config-load line; found {len(main_matches)}.')
main_idx = main_matches[0]
indent = lines[main_idx][:len(lines[main_idx]) - len(lines[main_idx].lstrip(' \t'))]
seed_lines = [
    indent + 'import random\n',
    indent + 'import numpy as np\n',
    indent + 'seed = int(getattr(cfg, "seed", 1))\n',
    indent + 'random.seed(seed)\n',
    indent + 'np.random.seed(seed)\n',
    indent + 'torch.manual_seed(seed)\n',
    indent + 'if torch.cuda.is_available():\n',
    indent + '\ttorch.cuda.manual_seed_all(seed)\n',
    indent + 'torch.backends.cudnn.deterministic = True\n',
    indent + 'torch.backends.cudnn.benchmark = False\n',
]
lines[main_idx + 1:main_idx + 1] = seed_lines
patched_train = ''.join(lines)

# Periodically save actor-only snapshots after the actor optimizer update. The final
# model is saved by the original end-of-training block below.
lines = patched_train.splitlines(keepends=True)
actor_loss_matches = [i for i, line in enumerate(lines)
                      if re.search(r'^\s*ave_act_loss\s*\+=\s*act_loss\.item\(\)', line)]
if len(actor_loss_matches) != 1:
    preview = '\n'.join(f'{i+1}: {line.rstrip()}' for i, line in enumerate(lines)
                        if 'act_optim.step' in line or 'act_lr_scheduler.step' in line or 'ave_act_loss' in line)
    raise RuntimeError(f'Expected one actor-loss accumulator line; found {len(actor_loss_matches)}. Relevant lines:\n{preview}')
insert_at = actor_loss_matches[0]
indent = lines[insert_at][:len(lines[insert_at]) - len(lines[insert_at].lstrip(' \t'))]
periodic_lines = [
    indent + 'completed_steps = i + 1\n',
    indent + "checkpoint_every = int(os.environ.get('TSP_DRL_CHECKPOINT_EVERY', '1000'))\n",
    indent + 'if cfg.issaver and checkpoint_every > 0 and completed_steps % checkpoint_every == 0 and completed_steps < cfg.steps:\n',
    indent + '\tcheckpoint_name = f"{cfg.task}_{date}_step{completed_steps}_act.pt"\n',
    indent + '\tcheckpoint_path = os.path.join(cfg.model_dir, checkpoint_name)\n',
    indent + '\ttorch.save(act_model.state_dict(), checkpoint_path)\n',
    indent + '\tprint(f"saved periodic actor checkpoint: {checkpoint_path}")\n',
]
lines[insert_at:insert_at] = periodic_lines
patched_train = ''.join(lines)

# Keep the upstream final-save statement unchanged. Repository versions can use
# different filename formatting, and renaming it is not needed for periodic snapshots.
# The upstream final save still runs at the end of training. Periodic snapshots above
# provide intermediate actor weights only (not a resumable optimizer/critic state).

# Save the applied patch and compile before training.
train_path.write_text(patched_train, encoding='utf-8')
patch_text = ''.join(difflib.unified_diff(
    upstream_train.splitlines(keepends=True), patched_train.splitlines(keepends=True),
    fromfile='a/train.py (upstream)', tofile='b/train.py (experiment patch)'))
(SHARED_ROOT / 'training_patch.diff').write_text(patch_text, encoding='utf-8')
print(patch_text)
print('\nPatched train.py syntax check...')
import py_compile
py_compile.compile(str(train_path), doraise=True)
print('PASS: train.py compiles.')
print('PASS: patch based on pristine pinned Git version; original upstream final-save statement preserved.')
for _run in TRAIN_RUNS:
    _root = RUN_ROOTS[_run['run_id']]
    shutil.copy2(SHARED_ROOT / 'train.py.upstream', _root / 'train.py.upstream')
    shutil.copy2(SHARED_ROOT / 'training_patch.diff', _root / 'training_patch.diff')


## Step 4 — Smoke test before the full runs

This is a separate 30-update test using batch size 32. It checks that the pinned code and documented patch execute on the selected GPU. It is not one of the scientific training runs and its outputs are not used in the thesis results.


In [ ]:
def prepare_training_config(run_dir, batch_size, steps, seed, log_step=100):
    run_dir = Path(run_dir)
    csv_dir, pt_dir, pkl_dir = run_dir / 'Csv', run_dir / 'Pt', run_dir / 'Pkl'
    for d in [csv_dir, pt_dir, pkl_dir]:
        d.mkdir(parents=True, exist_ok=True)
    pkl_path = pkl_dir / f'train{TRAIN_CITY_COUNT}.pkl'
    if pkl_path.exists():
        print('Using existing config:', pkl_path)
        return pkl_path
    cmd = [
        sys.executable, 'config.py', '-m', 'train',
        '-b', str(batch_size), '-t', str(TRAIN_CITY_COUNT), '-s', str(steps),
        '-e', '128', '-hi', '128', '-c', '10', '-st', '1.0',
        '-o', 'Adam', '-ng', '1', '-np', '3', '-dt', 'sampling',
        '--lr', '0.001', '--seed', str(seed), '-ls', str(log_step),
        '-ld', str(csv_dir) + os.sep,
        '-md', str(pt_dir) + os.sep,
        '-pd', str(pkl_dir) + os.sep,
    ]
    result = subprocess.run(cmd, cwd=REPO_DIR, text=True, capture_output=True, check=True)
    print(result.stdout)
    if result.stderr.strip():
        print(result.stderr)
    config_json = {
        'mode': 'train', 'city_t': TRAIN_CITY_COUNT,
        'batch': batch_size, 'steps': steps, 'seed': seed,
        'embed': 128, 'hidden': 128, 'clip_logits': 10,
        'softmax_T': 1.0, 'optim': 'Adam', 'n_glimpse': 1,
        'n_process': 3, 'decode_type': 'sampling', 'lr': 0.001,
        'lr_decay': 0.96, 'lr_decay_step': 5000,
        'fixed_step_budget': True, 'checkpoint_every': CHECKPOINT_EVERY,
    }
    (run_dir / 'config.json').write_text(json.dumps(config_json, indent=2), encoding='utf-8')
    if not pkl_path.exists():
        raise FileNotFoundError(f'config.py did not create expected config: {pkl_path}')
    return pkl_path

def run_training(run_dir, pkl_path, checkpoint_every=1000):
    run_dir = Path(run_dir)
    pt_dir = run_dir / 'Pt'
    # Prevent silently overwriting/duplicating an existing run. Change RUN_ID for a new run.
    if any(pt_dir.glob('*_step*_act.pt')):
        raise RuntimeError(f'Actor checkpoints already exist in {pt_dir}. Do not rerun training over them; use a new run ID for a genuinely fresh run.')
    env = os.environ.copy()
    env['TSP_DRL_CHECKPOINT_EVERY'] = str(checkpoint_every)
    log_path = run_dir / 'training_stdout.log'
    command = [sys.executable, 'train.py', '-p', str(pkl_path)]
    print('Command:', ' '.join(command))
    with log_path.open('w', encoding='utf-8') as log_file:
        proc = subprocess.Popen(command, cwd=REPO_DIR, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log_file.write(line)
            log_file.flush()
        code = proc.wait()
    if code != 0:
        raise subprocess.CalledProcessError(code, command)
    ckpts = sorted(pt_dir.glob('*_step*_act.pt'))
    print(f'\nTraining process finished. Actor checkpoint files: {len(ckpts)}')
    for p in ckpts:
        print(f'  {p.name} ({p.stat().st_size / (1024**2):.2f} MiB)')
    return ckpts

SMOKE_DIR = SHARED_ROOT / 'smoke_test'
smoke_pkl = prepare_training_config(SMOKE_DIR, batch_size=32, steps=30, seed=SMOKE_SEED, log_step=10)
smoke_ckpts = run_training(SMOKE_DIR, smoke_pkl, checkpoint_every=10)
print('Smoke-test artifacts:', SMOKE_DIR)


## Step 5 — Train the two additional models from random initialization

The runs execute sequentially to avoid competing for the same GPU. If either run is interrupted, preserve its folder and logs. Do not resume by loading its actor-only snapshots; a controlled resume would require optimizer, critic and RNG states, which these snapshots do not contain.

The expected coordinate tensor size is approximately 0.99 GiB per run before other model/runtime allocations. This is a resource note, not a runtime estimate.


In [ ]:
import re

expected_dataset_gib = TRAIN_BATCH_SIZE * TRAIN_STEPS * TRAIN_CITY_COUNT * 2 * 4 / (1024**3)
print(f'Approx. training-coordinate tensor size (float32): {expected_dataset_gib:.2f} GiB per run')
training_records = []

for run in TRAIN_RUNS:
    run_id, seed = run['run_id'], int(run['seed'])
    run_root = RUN_ROOTS[run_id]
    full_run_dir = run_root / 'full_train_n20'
    print('\n' + '=' * 72)
    print('Starting independent run:', run_id, '| seed:', seed)
    print('Output:', full_run_dir)
    full_pkl = prepare_training_config(full_run_dir, batch_size=TRAIN_BATCH_SIZE,
                                      steps=TRAIN_STEPS, seed=seed, log_step=100)
    checkpoints = run_training(full_run_dir, full_pkl, checkpoint_every=CHECKPOINT_EVERY)
    suffix_by_file = {}
    for checkpoint in checkpoints:
        match = re.search(r'_step(\d+)_act\.pt$', checkpoint.name)
        if match:
            suffix_by_file[checkpoint.name] = int(match.group(1))
    present_suffixes = set(suffix_by_file.values())
    required_suffixes = {1000, 5000, 10000, 12999}
    if not required_suffixes.issubset(present_suffixes):
        raise RuntimeError(f'{run_id}: missing required checkpoint suffixes {sorted(required_suffixes - present_suffixes)}. Preserve the run and inspect its log; do not report it as complete.')
    if not (run_root / 'training_patch.diff').is_file():
        raise FileNotFoundError(f'Missing archived patch for {run_id}')
    training_manifest = {
        **manifest_by_run_id[run_id],
        'commit_sha': COMMIT_SHA,
        'full_run_dir': str(full_run_dir),
        'training_steps_configured': TRAIN_STEPS,
        'training_steps_completed': 13000,
        'training_batch_size': TRAIN_BATCH_SIZE,
        'training_coordinate_tensor_estimated_gib': expected_dataset_gib,
        'checkpoint_suffix_semantics': 'suffix 12999 is the final saved actor after 13000 optimizer updates; the loop index starts at zero',
        'actor_checkpoints': [
            {'path': str(p), 'filename': p.name, 'suffix': suffix_by_file.get(p.name),
             'completed_optimizer_updates': (13000 if suffix_by_file.get(p.name) == 12999 else suffix_by_file.get(p.name)),
             'bytes': p.stat().st_size,
             'sha256': hashlib.sha256(p.read_bytes()).hexdigest()}
            for p in sorted((full_run_dir / 'Pt').glob('*_step*_act.pt'))
        ],
        'training_completed_at_local': datetime.now().isoformat(timespec='seconds'),
    }
    manifest_path = run_root / 'manifest_training.json'
    manifest_path.write_text(json.dumps(training_manifest, indent=2), encoding='utf-8')
    record = {
        'run_id': run_id, 'seed': seed,
        'manifest_path': str(manifest_path),
        'checkpoint_count': len(training_manifest['actor_checkpoints']),
        'final_checkpoint_present': 12999 in present_suffixes,
        'training_stdout_log': str(full_run_dir / 'training_stdout.log'),
    }
    training_records.append(record)
    print('Completed run manifest:', manifest_path)
    print('Checkpoint count:', record['checkpoint_count'])

extension_manifest = {
    'extension_id': 'multiseed_extension_20261025_v1',
    'main_run_id': MAIN_RUN_ID,
    'source_commit': COMMIT_SHA,
    'additional_training_seeds': TRAIN_SEEDS,
    'training_configuration': {
        'city_count': TRAIN_CITY_COUNT, 'batch_size': TRAIN_BATCH_SIZE,
        'optimizer': 'Adam', 'learning_rate': 0.001,
        'embedding_dimension': 128, 'hidden_dimension': 128,
        'optimizer_updates_per_run': TRAIN_STEPS,
        'checkpoint_suffixes_required_for_selection': [1000, 5000, 10000, 12999],
    },
    'runs': training_records,
    'warning': 'The notebook is only a protocol until each run has completed and its saved artifacts have been checked.'
}
extension_manifest_path = SHARED_ROOT / 'multiseed_training_manifest.json'
extension_manifest_path.write_text(json.dumps(extension_manifest, indent=2), encoding='utf-8')
print('\nTRAINING EXTENSION SUMMARY')
print(json.dumps(extension_manifest, indent=2))
